In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
import numpy as np
import pandas as pd
from pathlib import Path
import os

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import datasets, layers, models, losses, Model
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

import subprocess
from IPython.display import FileLink, display

from sklearn.metrics import confusion_matrix, classification_report

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
# Count number of training images for both classes to calculate a
# data-driven training batch size.
num_samples = (len(os.listdir('data_small/Chic')) +
               len(os.listdir('data_small/Duck')))

# We use 200 batches.
img_height, img_width = 224,224
batch_size = num_samples // 200

In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
train_ds = tf.keras.utils.image_dataset_from_directory(
  'data_small',
  validation_split=0.2,
  subset="training",
  label_mode='binary',
  seed=123, #number to randomize outcome
  image_size=(img_height, img_width),
  batch_size=batch_size)

Found 735 files belonging to 2 classes.
Using 588 files for training.


In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
val_ds = tf.keras.utils.image_dataset_from_directory(
 'data_small',
  validation_split=0.2,
  subset="validation",
  label_mode='binary',
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

Found 735 files belonging to 2 classes.
Using 147 files for validation.


In [5]:
# --- [CELL 4]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
test_ds = tf.keras.utils.image_dataset_from_directory(
 'data_small_test',
  image_size=(img_height, img_width),
  label_mode='binary',
  batch_size=batch_size)

Found 146 files belonging to 2 classes.


In [6]:
# --- [CELL 5]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
base_model = tf.keras.applications.ResNet50(weights = 'imagenet', include_top = False, input_shape = (224,224,3))

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [7]:
# --- [CELL 6]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 7}
x = base_model.output
x = keras.layers.GlobalAveragePooling2D()(x)

# 1024 neurons are half the 2048 output dimensionality of the previous
# layer and the last layer from base ResNet-50.
x = keras.layers.Dense(units=1024, activation='relu')(x)
x = keras.layers.Dense(units=512, activation='relu')(x)
x = keras.layers.Dense(units=256, activation='relu')(x)
x = keras.layers.Dense(units=128, activation='relu')(x)
x = keras.layers.Dense(units=1, activation='sigmoid')(x)

model = keras.models.Model(inputs=base_model.input,
                                    outputs=x)

In [8]:
# --- [CELL 7]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 8}
for layer in model.layers[:175]:
    layer.trainable = False

In [9]:
# --- [CELL 8]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 9}
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [10]:
# --- [CELL 9]: ---
# cell_state: unchanged
# execution_status: {'status': 'error', 'done': True, 'execution_count': 10}
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=2, #100,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            patience=3
        )
    ]
)

Epoch 1/2
196/196 ━━━━━━━━━━━━━━━━━━━━ 29s 119ms/step - accuracy: 0.7884 - loss: 0.4559 - val_accuracy: 1.0000 - val_loss: 3.1740e-05 - learning_rate: 0.0010
Epoch 2/2
196/196 ━━━━━━━━━━━━━━━━━━━━ 19s 96ms/step - accuracy: 1.0000 - loss: 1.2369e-06 - val_accuracy: 1.0000 - val_loss: 1.6796e-06 - learning_rate: 0.0010


In [11]:
# --- [CELL 10]: ---
# cell_state: unchanged
# execution_status: {'status': 'not run'}
results = model.evaluate(test_ds, verbose=0)
print("    Test Loss: {:.5f}".format(results[0]))
print("Test Accuracy: {:.2f}%".format(results[1] * 100))

    Test Loss: 0.00019
Test Accuracy: 100.00%


In [12]:
# --- [CELL 11]: ---
# cell_state: unchanged
# execution_status: {'status': 'not run'}
predictions = (model.predict(test_ds) >= 0.5)

49/49 ━━━━━━━━━━━━━━━━━━━━ 6s 93ms/step


In [13]:
# --- [CELL 12]: ---
# cell_state: edited
# execution_status: {'status': 'not run'}
# === BEFORE (original) ===
# predictions = np.array([])
# labels =  np.array([])
# for x, y in test_ds:
#   predictions = np.concatenate([predictions, model.predict_classes(x)])
#   labels = np.concatenate([labels, np.argmax(y.numpy(), axis=-1)])
# 
# tf.math.confusion_matrix(labels=labels, predictions=predictions).numpy()

# === AFTER (edited) ===
predictions = np.array([])
labels =  np.array([])
for x, y in test_ds:
  preds = model.predict(x)
  preds = (preds >= 0.5).astype(int).flatten()  # Convert sigmoid outputs to binary class predictions
  predictions = np.concatenate([predictions, preds])
  labels = np.concatenate([labels, y.numpy().flatten()])  # Labels are already 0 or 1

tf.math.confusion_matrix(labels=labels, predictions=predictions).numpy()

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━

array([[73,  0],
       [ 0, 73]], dtype=int32)

In [14]:
import numpy as np
import tensorflow as tf

# Contract: previous cell must produce predictions artifact
assert 'predictions' in globals(), "Expected `predictions` from the previous cell."

pred = np.asarray(predictions)

# Normalize predictions to 1D class ids without assuming how they were produced.
# Accepts bool masks, class-id vectors, sigmoid probs, or one-hot/prob matrices.
if pred.ndim == 0:
    pred = pred.reshape(1)

if pred.ndim >= 2:
    if pred.shape[-1] == 1:
        pred = pred.reshape(-1)
    else:
        pred = np.argmax(pred, axis=-1).reshape(-1)
else:
    pred = pred.reshape(-1)

if pred.dtype == np.bool_:
    pred = pred.astype(np.int32)
elif np.issubdtype(pred.dtype, np.floating):
    # If float vector, treat as binary probs unless already near class ids.
    unique_rounded = np.unique(np.round(pred))
    if np.all(np.isin(unique_rounded, [0.0, 1.0])) and np.all((pred >= 0.0) & (pred <= 1.0)):
        pred = (pred >= 0.5).astype(np.int32)
    else:
        pred = pred.astype(np.int32)
else:
    pred = pred.astype(np.int32)

# Build true labels from test_ds robustly for binary or one-hot labels.
true_labels = []
for _, y_batch in test_ds:
    y = y_batch.numpy()
    if y.ndim >= 2 and y.shape[-1] > 1:
        y = np.argmax(y, axis=-1)
    else:
        y = y.reshape(-1)
    true_labels.append(y.astype(np.int32))

true_labels = np.concatenate(true_labels) if len(true_labels) else np.array([], dtype=np.int32)

# Core behavioral assertions
assert pred.ndim == 1 and true_labels.ndim == 1, "Predictions and labels must be 1D."
assert pred.shape[0] == true_labels.shape[0] > 0, (
    f"Prediction count ({pred.shape[0]}) must match label count ({true_labels.shape[0]}), and be > 0."
)
assert np.isfinite(pred).all(), "Predictions contain non-finite values."
assert np.isfinite(true_labels).all(), "Labels contain non-finite values."

# Class-id validity (binary problem in this notebook)
assert set(np.unique(true_labels)).issubset({0, 1}), f"Unexpected label ids: {np.unique(true_labels)}"
assert set(np.unique(pred)).issubset({0, 1}), f"Unexpected prediction ids: {np.unique(pred)}"

cm = tf.math.confusion_matrix(labels=true_labels, predictions=pred, num_classes=2).numpy()
assert cm.shape == (2, 2), f"Expected 2x2 confusion matrix, got {cm.shape}"
assert int(cm.sum()) == int(true_labels.shape[0]), "Confusion matrix count must equal number of samples."